In [ ]:
import os
# manually specify the GPUs to use
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0"
import sys
import torch
import tqdm 
import pickle as pkl
import numpy as np


module_path = os.path.abspath('..')
if module_path not in sys.path:
    sys.path.insert(0, module_path)

from torch.utils.data import DataLoader
from datasets import TransformerConf4Dataset, CNFDataset
from models import TransformerConf3, LightningModelCNF
from utils import args_transformer, args_cnf, create_mask_src, create_mask_tgt

from config import FitConfig
from model_loader import GlobalModelLoader
from run_fit import VertexFitter



device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.multiprocessing.set_sharing_strategy('file_system')

cfg = FitConfig(
    data_term="poisson",
    lam_a = 1e-5,
    adam_lr = 5e-2,
    adam_steps = 20,
    background_mode="none",

    n_sample_for_template=1,
)


In [2]:
# set up the models
model_loader = GlobalModelLoader(device)
model_loader.load_models()





- Arguments:
  dataset_path: /pscratch/sd/b/botaoli/SFGD_VA/Data/NN_Data_compressed/{}/{}/{}/{}.zip
  metadata_path: /pscratch/sd/b/botaoli/SFGD_VA/Data/NN_Data_compressed/metadata.pkl
  event_in_folder: 100000
  seed: 42
  max_p: 5
  max_p_contained: 5
  max_p_exiting: 1
  mom_smearing_p: 0.07
  mom_smearing_mu: 0.07
  cube_size: 10.27
  va_size: 7
  pad_value: -10000
  hidden: 192
  dropout: 0.1
  encoder_layers: 10
  decoder_layers: 10
  attn_heads: 16
  batch_size: 16
  epochs: 12420
  num_workers: 32
  lr: 0.002
  accum_grad_batches: 4
  cosine_annealing_steps: 400
  weight_decay: 0.01
  beta1: 0.9
  beta2: 0.999
  eps: 1e-09
  warmup_steps: 20
  save_dir: logs
  name: v1
  log_every_n_steps: 50
  early_stop_patience: 0
  save_top_k: 1
  checkpoint_path: /pscratch/sd/b/botaoli/SFGD_VA/Results/checkpoints
  checkpoint_name: v4
  load_checkpoint: None
  gpus: [0]
  num_nodes: 1


/global/homes/b/botaoli/.conda/envs/env_nn/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Model weights loaded!


In [3]:
# for test
test_data = model_loader.load_sample_data(0)

Number of batches: 7813


  0%|          | 0/7813 [00:00<?, ?it/s]/global/homes/b/botaoli/.conda/envs/env_nn/lib/python3.11/site-packages/torch/nn/functional.py:6041: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
  0%|          | 0/7813 [00:01<?, ?it/s]


In [4]:
print("true_ekin: ", test_data["ekin_true"][0])
print("true_dir: ", test_data["dir_true"][0])
print("true_vtx: ", test_data["vtx_true"][0])
print("pred_ekin: ", test_data["ekin_pred"][0])
print("pred_dir: ", test_data["dir_pred"][0])
print("pred_vtx: ", test_data["vtx_pred"][0])
print("N_pred: ", test_data["N_pred"][0])
print("N_true: ", test_data["N_true"][0])



true_ekin:  [[ 1.1906271]
 [-0.8727242]
 [-1.1738924]
 [-1.4489642]]
true_dir:  [[-0.02747619  0.9966424  -0.0771303 ]
 [-0.29660094 -0.2181903  -0.9297424 ]
 [ 0.8608451  -0.49747765  0.10705991]
 [-0.69898605 -0.49889147 -0.51237273]]
true_vtx:  [ 0.02203437 -0.26305288 -0.25678235]
pred_ekin:  [[ 1.3642532]
 [-0.8173955]
 [-1.3034853]]
pred_dir:  [[-0.09449544  0.9839253  -0.15153117]
 [-0.24717224  0.4369957  -0.8648356 ]
 [-0.15659338  0.9684246   0.1939904 ]]
pred_vtx:  [ 0.25738963 -0.2833644  -0.21387681]
N_pred:  3
N_true:  4


In [5]:
data = test_data["hits"][0]
#convert to a tensor
data = torch.tensor(data)
fitter = VertexFitter(model_loader, cfg)
#result = fitter.fit(data, 0)


In [6]:
result = fitter.fit(data, 0)

Number of batches: 7813


  0%|          | 0/7813 [00:00<?, ?it/s]

  0%|          | 0/7813 [00:00<?, ?it/s]


tensor(1., device='cuda:0')
TrackGenerator parameters shape:  torch.Size([1, 7])
TrackGenerator parameters:  tensor([[ 0.2574, -0.2834, -0.2139,  1.6388, -0.7492,  0.4802,  0.0246]],
       device='cuda:0', grad_fn=<UnsqueezeBackward0>)
generated_voxels shape:  torch.Size([4, 125])
generated_voxels:  tensor([[7.6509e-01, 1.3729e+00, 2.3225e-01, 1.6975e-01, 8.4054e-02, 4.1804e-01,
         2.5248e-01, 1.2301e+00, 1.5456e-01, 1.5392e+00, 2.4787e-02, 1.1274e+00,
         1.0941e+00, 1.3812e+00, 8.8040e-01, 8.8826e-01, 4.1460e-01, 1.6406e+02,
         1.2429e+00, 7.7178e-01, 9.5213e-01, 1.1324e+00, 1.8584e+01, 3.8727e-02,
         5.6904e-01, 9.7535e-03, 6.1218e-01, 1.0541e+00, 1.3722e+00, 2.7022e-01,
         6.6786e-01, 1.1276e+00, 1.4672e+00, 7.7161e-01, 1.1371e+00, 1.4803e+00,
         1.2114e+00, 1.1855e+02, 4.0292e-01, 5.8589e-01, 1.2526e+00, 1.2600e-01,
         4.0920e+01, 4.5730e-01, 2.7471e-01, 1.4362e+00, 1.5578e+00, 9.3438e-01,
         2.1646e-01, 4.3837e-02, 5.7509e-01, 4.483

UnboundLocalError: cannot access local variable 'drift_loss' where it is not associated with a value

In [ ]:
print(test_data.keys())

In [ ]:
print(test_data['exit_particle'])

